# How a knee MRI becomes a tensor

Every study arrives different: a different number of sequences, a different number of
slices, a different matrix size, a different pixel spacing, a different intensity
scale, and a knee that may be the left or the right one. The model needs the same
thing every time.

This notebook is the story of how that gap is closed, and — just as important —
**whose idea each step was**. Almost all of it is ported from one public notebook. The
reasoning is often good, sometimes measured, occasionally just a convention nobody has
tested. Each section says which.

> **Read this as a set of decisions, not a specification.** Everything here is one
> team's answer to a problem that has other answers. Where we have measured something
> ourselves it is marked; where we are repeating a claim, it is attributed; where
> nobody has checked, it says so.

The output, always:

```
tensor  (6, 3, 336, 336)  uint8      +      presence mask  (6,)
```

Six slots, three slices each, 336 pixels square, bytes. Plus a mask saying which of the
six actually exist.

---

**Nothing is implemented here.** Every table and every figure comes from `rsna.viz`,
and `verify_against_read_slot` asserts the illustrated chain ends exactly where the real
pipeline ends — so this cannot drift into describing a preprocessing nobody runs.

Needs DICOMs under `data/raw/<split>/<study>/<series>/*.dcm`. Outputs are stripped on
commit (`scripts/nbstrip.py`): rendered slices are competition data.

## Where this comes from

Nearly every choice below is ported from **pilkwang's** public baseline notebook,
vendored unmodified at
[`notebooks/external/rsna-knee-baseline-v1/rsna-knee-baseline-v1.ipynb`](../notebooks/external/rsna-knee-baseline-v1/rsna-knee-baseline-v1.ipynb).
Its prose sections argue each decision before the code, and most of the quotations here
are its docstrings. Cell numbers are given per section.

Two other notebooks are vendored, and both matter for judging how settled any of this is:

| Notebook | What it tells us |
|---|---|
| [`rsna-knee-read-the-report-then-the-knee`](../notebooks/external/rsna-knee-read-the-report-then-the-knee/rsna-knee-read-the-report-then-the-knee.ipynb) (prvsiyan) | Carries **32 of these functions byte-identical**. Two independent top scorers agreeing is the closest thing this competition has to a consensus. |
| [`bend-the-knee-to-the-dinosaurs`](../notebooks/external/bend-the-knee-to-the-dinosaurs/bend-the-knee-to-the-dinosaurs.ipynb) (mattiaangeli) | The same pipeline with every explanatory comment stripped, plus ensemble stacking. Useful as evidence of what the top of the leaderboard is made of, not as a source of reasoning. |

Licences and full provenance: [`references.md`](references.md). The vendored copies are
never edited — the day we change something, we own a copy.

The output, always:

```
tensor  (6, 3, 336, 336)  uint8      +      presence mask  (6,)
```

Six slots, three slices each, 336 pixels square, bytes. Plus a mask saying which of
the six actually exist.

---

---

## Setting up

One header per series, for whatever DICOMs are on disk.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
%matplotlib inline

from rsna import viz
from rsna.config import Config
from rsna.dicom import annotate, walk

DATA = ROOT / "data/raw"
SPLIT = "test_series"

config = Config()
headers = annotate(walk(DATA, SPLIT))
plane_map = viz.plane_map_from(DATA, SPLIT, headers)
headers["plane"] = headers["SeriesInstanceUID"].map(plane_map)

print(f"{config.img}px, {config.slices} slices cached, {config.group} per window, "
      f"{config.n_slot} slots")

---

## Problem 1 — a study has between 3 and 14 sequences

**The fact** (measured on the 4,407 training studies):

```
 3 series :     1 study
 4 series :   675
 5 series :  2299   <- the great majority
 6 series :   698
 7 series :   310
 ...
14 series :     1
```

Twenty-two sites contributed, each with its own habits. There is no standard protocol.

**Their solution** (baseline notebook, cell 13 for the list, cell 16 for `pick_slots`).
Six fixed *slots*, each a predicate on (plane, fluid-sensitive, fat-suppressed):

```python
("SAG_FLUID_FS",   "Sagittal", True,  True)
("COR_FLUID_FS",   "Coronal",  True,  True)
("AX_FLUID_FS",    "Axial",    True,  True)
("SAG_FLUID_NOFS", "Sagittal", True,  False)
("COR_T1",         "Coronal",  False, False)
("SAG_T1",         "Sagittal", False, False)
```

Every acquired series is tested against the six. If several match a slot, the one with
the **most slices** wins — a thicker stack samples the joint more densely. Series
matching nothing are discarded.

**Why six.** This is a structural compromise, not a list of "the useful sequences". A
bag of variable size has to become a tensor of fixed size, and every extra slot costs
one encoder pass per study, at training *and* at inference. The six cover the three
planes in fluid-sensitive fat-suppressed sequences — where effusion, oedema and acute
lesions show — plus two structural T1 slots, where anatomy and osteoarthritis show,
plus one fluid sagittal without fat suppression.

`AX_FLUID_NOFS` is deliberately absent: on the delivered flags, axial non-fluid exists
for only **19.4%** of studies (our measurement). A slot that is empty four times out of
five carries little and costs the same as any other.

**What it costs.** On our three local studies, 15 acquired series produce 12 filled
slots — **3 series are thrown away**. One lost a tie; two matched no slot at all. We
cannot yet measure this rate over the full corpus, because `fatsat` and `fluid` are
recovered from DICOM headers we do not have locally.

**Untested.** Whether six is the right number, whether "most slices" is the right
tie-break, and whether the discarded 20% carries anything. Nobody has published an
ablation on any of it.

---

---

## Problem 2 — the competition's two flags are really one

**The fact** (our measurement): `Fluid_Sensitive` and `Fat_Suppression` are shipped per
series, and they are **equal on all 24,371 training rows**. As delivered they carry one
axis, not two.

**Their solution** (baseline notebook, cell 15, `annotate`). Recover both from the
DICOM header instead: `SeriesDescription`,
`SequenceName`, `ScanOptions`, and — when the text says nothing — `RepetitionTime` and
`EchoTime`.

This is what makes six *distinct* slots possible rather than six slots where half are
duplicates. It is probably the single best idea in the pipeline, and it is invisible if
you only read the CSVs.

**One trap it handles.** GE writes `SAT_GEMS` in `ScanOptions` for *spatial* saturation,
which is not fat suppression. A substring test on "SAT" would misclassify those series,
so scan options are matched as exact tokens. We saw this fire on real data: `Sag PD FSE`
carries `SAT_GEMS` and is correctly **not** fat-suppressed.

**A limit.** One study in our three has no `ScanOptions`, no `RepetitionTime` and no
`EchoTime` at all — the headers were stripped differently by that site. It is classified
from the description alone (`SG T2W FS` -> T2, fat-sat). That works here; it is a
heuristic, not a guarantee.

---

**Seen on these series.** `weight` and `fatsat` are reconstructed from the description, the scan options and, when the text says nothing, TR and TE.

Look for `SAT_GEMS` in the scan options: GE writes it for *spatial* saturation, which is not fat suppression. A substring test on "SAT" would misclassify those series.

In [ ]:
viz.report_headers(headers)

---

## Problem 3 — left knees and right knees

**The fact.** Five of the twelve targets are named for a side: the two menisci, the two
tibiofemoral compartments, and the medial collateral ligament. Medial and lateral are
defined relative to the body midline, so which side of the *image* they fall on depends
on which knee was scanned.

And `Laterality` (0020,0060) is Type 2C — it may legitimately be absent. Their docstring
says it is missing on **exactly half** the studies, by whole vendors rather than
scattered series. On our three studies: one tagged, two recovered from geometry, one of
which had the tag present as an **empty string** rather than absent.

**Their solution** (baseline notebook, cell 15, `side_from_geometry` and `lat_of`;
cell 21 for the mirror). Tag where it exists; geometry where it does not. DICOM patient
coordinates are LPS, so +x is the patient's left and the centre of a right knee sits at
negative x. The **centre** is used rather than `ImagePositionPatient` itself, which is
the image corner and is offset by half a field of view — enough to flip the sign on a
knee near the midline. Then mirror every right knee onto a left-knee convention.

**Where it stops.** Studies whose centre falls within 20 mm of the midline are left
**unresolved** rather than guessed: measured against the tagged half, the geometric rule
is right 97% of the time overall and *no better than chance inside 20 mm*. An unresolved
study is left unmirrored — a decision, not a default.

**Why sagittal slices are not mirrored.** Their left-right image axis is the
through-plane direction. Mirroring a sagittal stack is done by reversing slice order,
not by flipping pixels — which is another reason problem 5 matters.

---

In [ ]:
sides = viz.report_laterality(headers, config)

---

## Problem 4 — some sequences were never acquired

**The fact.** On our three studies, **12 of 18 slots are filled**. One study has no T1
at all and no sagittal fat-sat.

**Their solution** (baseline notebook, cell 16 for the slot, cell 25 for the head). A
slot with no matching series **stays empty**, and a presence mask records it. The head then excludes it from its softmax rather than pooling it:

```python
att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
```

**Why not fill it.** A black image is a picture of something; absence is not. Filling an
empty slot with zeros would tell the model "this sequence was acquired and shows
nothing", which is false.

Their notebook also rejects a softer version — relaxing the weighting to fill an empty
structural slot from a neighbouring predicate — with a measured argument: over the
training corpus it would put **one series in two slots for 2,383 of 4,407 studies**, and
leave 56% of the T1 slot holding PD or T2. The per-diagnosis attention would then split
across two identical slots, giving one acquisition roughly twice the weight it carries
in a study that holds both.

**An inefficiency we found.** `Model.forward` reshapes *all* six slots into the encoder
before the mask is ever consulted. On our three studies that means **a third of the
encoder passes are spent on black images**. It changes no prediction — only speed — and
it is present in their code too. Fixable; not yet fixed.

---

**Seen on these studies.** Which acquired series filled which fixed position — and which were discarded because they matched no slot, or lost the tie-break on slice count.

In [ ]:
slots = viz.report_slots(headers, plane_map, config)

---

## Pick one (study, slot)

Everything below is about a single series. Change these two lines and re-run.

In [ ]:
STUDY = None      # or the last characters of a StudyInstanceUID, e.g. "7456991776"
SLOT = None       # or a slot name, e.g. "SAG_FLUID_FS"

selection = viz.selection_of(slots, limit=4, study=STUDY, slot=SLOT)
study, slot, record = selection[0]
side = sides.get(study)

viz.verify_against_read_slot(record, config)
print(f"{slot}  {record['SeriesDescription']!r}  {record['n_slices']} slices  "
      f"{record['px']:.3f} mm/px  side {side or 'unresolved'}")
print("illustrated chain matches read_slot exactly")

---

## Problem 5 — file order is not slice order

**The fact.** A DICOM filename here is a SOP Instance UID: assigned to be *unique*,
ordered by nothing. Spearman correlation between filename order and physical order, on
four real series:

```
-0.092   -0.142   -0.274   +0.086
```

Noise.

**Their solution** (baseline notebook, cell 19, `order_slices`; argued in its §3b
markdown). Sort geometrically: project `ImagePositionPatient` onto the slice
normal derived from `ImageOrientationPatient`; fall back to `InstanceNumber` when the
geometry is absent; keep the arbitrary order and *count it* when nothing works.

**Why it matters more than it looks.** The three channels of a 2.5D input are supposed
to be neighbouring slices. If the order is noise, they are three unrelated views, "the
middle of the stack" is a random subset, and reversing slice order to normalise
laterality reverses nothing.

**And it fails silently.** No exception, no warning — only a model that underperforms,
which is indistinguishable from a hard problem. This is the trap most likely to cost a
newcomer weeks.

---

In [ ]:
viz.report_ordering(selection, config)

**And in pixels.** Top row: the three channels in geometric order, and what they make as one RGB image. Bottom row: the same, ordered by filename — what a pipeline that sorts by name would feed the model.

Watch the composites. Neighbouring slices agree, so the RGB stays close to grey with thin fringes at edges. Arbitrary slices do not, and the colour across the whole image *is* the third dimension carrying noise — with no error raised anywhere.

In [ ]:
fig = viz.figure_channels(record, config, side)

---

## Problem 6 — a series has between 24 and 160 slices

**The fact** (our three studies): 24, 25, 30, 33, 34, 40, 160.

**Their solution** (baseline notebook, cell 19, `read_slot`). Always take the same
*number* of slices — `config.slices`, which is
12 for the published checkpoints — spread over the **central 20–80%** of the stack:

```
 24 slices -> [4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16, 18]
160 slices -> spread from 31 to 127
```

Always the same fractions, never the same indices.

**Why a central band.** The outermost slices of a knee series are mostly soft tissue
outside the joint. Their docstring is honest about the trade-off: *"how much of the
stack is worth reading depends on how many slices are being taken — at three the middle
is all that fits, while at sixteen the ends are worth having, and a Baker cyst sits at
the posteromedial end of a sagittal one."*

So the 20–80% band is a compromise forced by taking few slices, and they say so. **If a
Baker's cyst sits in the excluded 20%, this pipeline cannot see it.**

**What we measured.** The 12 cached slices are *not* redundant with each other. Pixel
correlation between cached slices, by separation:

| Separation | Correlation |
|---|---|
| 1 position (~4 mm) | 0.348 |
| 2 positions | 0.384 |
| 6 positions | 0.355 |
| 11 positions (~56 mm) | 0.317 |

**It is flat.** Two adjacent cached slices resemble each other about as much as two from
opposite ends of the knee. At 4 mm apart, the anatomy has already changed completely.

That cuts both ways: the usual justification for stacking *adjacent* slices — capturing
local 3-D continuity — does not survive this measurement either. What is left is that
three slices are what a three-channel pretrained encoder accepts. **It is a convention
inherited from the shape of the encoder, not a result.**

Caveat: crude pixel correlation on 12 slot-series from 3 studies. It measures overall
resemblance, not whether lesion-relevant signal is continuous.

---

**Seen on this series.** Every slice in physical order; red marks the ones the cache keeps. Note how much is left out.

In [ ]:
fig = viz.figure_stack(record, config)

---

## Problem 7 — the same knee at different physical scales

**The fact** (our three studies): matrices from 512x512 to 960x960 and 640x1280; pixel
spacing from **0.1562 to 0.3516 mm** — a factor of 2.3 within three studies. Their
docstring reports 3.4x across the full corpus.

**Their solution** (baseline notebook, `CROP_MM` in cell 13, applied in cell 19;
argued in its §4 markdown). Crop a constant number of **millimetres**, then resize to a
constant number of pixels:

```
pd_tse_tra_d   960x960 @ 0.156 mm/px  ->  crop 832 px  ->  336 px
Cor T1 SE      512x512 @ 0.352 mm/px  ->  crop 370 px  ->  336 px
```

The crop size in pixels varies; the physical extent does not. After the resize, one
millimetre of knee occupies the same number of pixels everywhere: `130/336 = 0.387 mm`.

**Why 130 mm.** Measured, and this one is worth quoting: the acquired field of view has
median 160 mm and runs 70–320, so *"a 160 mm crop is larger than the image in 60% of
series and is skipped for all of them"*. 130 mm is below the field of view of **99.6%**
of series and still contains the joint.

**The limit nobody discusses.** The crop is centred on the **image**, not on the joint.
It assumes the radiographer centred the knee. A Baker's cyst is posterior; if the
framing is off, the crop can cut it.

**And a hard floor underneath.** A feature narrower than two pixels cannot survive
sampling. At 0.387 mm/pixel that is 0.77 mm — and DINOv2's patches are 14 pixels, so
**one patch token covers 5.4 mm of knee**. Thin cartilage lesions live close to that
limit.

---

In [ ]:
viz.report_pixels(selection, sides, config)

---

## Problem 8 — MRI intensity has no absolute scale

**The fact.** A raw slice in our data runs from -560 to 7137. The number 1,500 means
nothing on its own: it depends on the scanner, the coil, the gain.

**Their solution** (baseline notebook, cell 19, end of `read_slot`). Clip to the
1st–99th percentile **of the whole stack**, rescale into
[0,1], quantise to bytes. Then, inside the model, a second normalisation — ImageNet's
`mean=[0.485, 0.456, 0.406]` — because that is what DINOv2 was trained with.

**Why percentiles rather than min/max.** A single bright vessel would otherwise compress
the entire dynamic range and flatten everything else.

**Why over the whole stack rather than per slice.** A nearly empty slice normalised on
its own would be stretched across the full range and look as contrasted as one full of
anatomy.

**Why bytes.** These buffers queue between reader threads and the encoder; at this size a
float32 slot-series is several megabytes. Intensity is already in [0,1], so eight bits
cost nothing that a bilinear resize has not already cost, and the queue is a quarter the
size.

---

**Problems 3, 7 and 8 on one slice.** Raw → cropped to a constant 130 mm of anatomy → normalised by the stack's 1st–99th percentile → resized → quantised to bytes → mirrored if this is a right knee.

The crop is in **millimetres**, so its size in pixels differs per series while the physical extent does not. That is what makes two series comparable.

In [ ]:
fig = viz.figure_steps(record, config, side)

---

## The cache, and the windows

`config.slices` slices are decoded and kept per slot — 12, which is what the published
checkpoints were fitted with. **The encoder still takes `config.group` = 3 at a time.**

How those 3 are chosen differs between the two regimes:

| | Windows | How many per pass | Why |
|---|---|---|---|
| Training | 4 disjoint: `[0, 3, 6, 9]` | **one**, drawn at random each step | augmentation along the stack |
| Inference | 10 overlapping: `[0..9]` | **all**, logits averaged | averages the randomness away |

Stochastic while fitting, averaged at test — the same idea as dropout. Both lists come
from `Config.windows`, so they travel with the weights and training and inference cannot
disagree about them.

In [ ]:
fig = viz.figure_cache(record, config, side)

### The four windows a training step can draw

One of these per step, at random. The model never sees two at once, so it learns that the finding does not depend on which window it got.

In [ ]:
fig = viz.figure_windows(record, config, side, overlap=False)

### The ten windows inference averages over

Sliding one slice at a time. Every one is a full encoder pass — which is why 20 members x 10 windows x 6 slots = 1200 passes per study, and why a scored run takes hours.

Note the offsets 1, 2, 4, 5, 7, 8: the model was **never trained on those**. Neighbouring slices are similar enough that it does not matter, but it is a deliberate asymmetry.

In [ ]:
fig = viz.figure_windows(record, config, side, overlap=True, max_rows=10)

---

## What is deliberately *not* normalised

| | Why it is left alone |
|---|---|
| **Contrast between sequences** | A T1 and a T2 fat-sat have opposite contrast, and both are handed to the same encoder as-is. The *slot* is what carries "which sequence this is" |
| **Bias field** | No N4 correction, standard though it is in MRI |
| **Registration between series** | The six slots are not aligned to each other |
| **Slice spacing** | The band is proportional, so the gap between sampled slices varies from ~4 to ~6 mm depending on the series |

None of these is an oversight — each would cost compute, and nobody has shown any of
them pays. They are open questions, and they are ours to test.

---

---

## The whole path, in order

```
walk            one header per series
annotate        recover fat-sat and weighting from the header      (problem 2)
laterality_of   tag, else geometry, else unresolved                 (problem 3)
pick_slots      map a variable bag onto 6 fixed positions           (problems 1, 4)
order_slices    sort by geometry, not by filename                   (problem 5)
read_slot       sample the band, crop mm, percentile, resize, byte  (problems 6, 7, 8)
normalise_...   mirror right knees                                  (problem 3)
build_cache     all of the above, once, into (n, 6, 12, 336, 336)
```

Every stage above appears in this notebook in that order, with what it decided on the
studies you have on disk. The code is in
[`src/rsna/dicom/`](../src/rsna/dicom/); the traps that make each stage necessary are
collected in [`docs/pipeline_pitfalls.md`](../docs/pipeline_pitfalls.md).


---

## What to remember

Three of these steps — geometric slice order, laterality, millimetre scaling — fix
problems that **produce no error when they are wrong**. The pipeline runs, the shapes
agree, the submission is well formed, and the model is simply worse. That is the
category of bug this preprocessing exists to prevent, and it is why the reasoning is
recorded here rather than left in the code.

And the rest is one team's judgement — pilkwang's, in
[`notebooks/external/rsna-knee-baseline-v1/`](../notebooks/external/rsna-knee-baseline-v1/),
seconded by prvsiyan on the parts they share verbatim. Six slots, a 20–80% band, 130 mm, three channels,
twelve cached slices: every one of those is a number someone chose. They are a good
starting point precisely because they are *documented* — not because they are right.